# Conformal Demand Risk — demo notebook

Walkthrough: load data → lag features → Ridge + split conformal → coverage vs nominal.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

from conformal_demand.data import load_wide, temporal_split_mask
from conformal_demand.features import feature_matrix_for_item
from conformal_demand.models import make_estimator
from conformal_demand.conformal import SplitConformalRegressor
from conformal_demand.metrics import point_metrics, interval_metrics


In [ ]:
wide = load_wide('raw')  # or 'sample'
X, y = feature_matrix_for_item(wide, 'item_1')
tr, ca, te = temporal_split_mask(len(X))
conf = SplitConformalRegressor(make_estimator('ridge'), alpha=0.10)
conf.fit(X.iloc[tr], y.iloc[tr], X.iloc[ca], y.iloc[ca])
res = conf.predict_interval(X.iloc[te])
print(point_metrics(y.iloc[te], res.y_pred))
print(interval_metrics(y.iloc[te], res.lower, res.upper, 0.90))
